# 🔍 RAG シミュレーション
APIキー不要

## 構成
```
ユーザー (httpx)
  ↕ HTTP POST /ask
RAGサーバー (FastAPI)
  ├─ RDB検索     (SQLite)   → 価格・カテゴリ等の構造化フィルタ
  └─ ベクトルDB検索 (ChromaDB) → 説明文の意味検索
       ↕ 両結果をマージしてプロンプトに組み込む
  ↕ HTTP POST /v1/chat/completions
LLMサーバー (llama-cpp-python)
  └─ Qwen2.5-0.5B-Instruct GGUF
```

## データ題材: 商品カタログ
| コンポーネント | 役割 |
|---|---|
| SQLite (RDB) | 商品名・価格・カテゴリ・在庫の構造化検索 |
| ChromaDB (ベクトルDB) | 商品説明文の意味検索 |
| FastAPI | RAGサーバー (`POST /ask`, `GET /products`) |
| httpx | LLMサーバーへの直接HTTP呼び出し |

> 💡  GPU 推奨

## Step 1 — インストール

In [ ]:
import subprocess, sys
has_gpu = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print("GPU 検出" if has_gpu else "CPU のみ")

if has_gpu:
    !CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python[server] \
        --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121 -q
else:
    !pip install llama-cpp-python[server] -q

!pip install chromadb sentence-transformers \
             fastapi uvicorn httpx huggingface_hub nest_asyncio -q
print("\n✅ インストール完了")

## Step 2 — モデルのダウンロード

In [ ]:
from huggingface_hub import hf_hub_download
import os

MODEL_DIR = "/content/models"
os.makedirs(MODEL_DIR, exist_ok=True)

print("📥 Qwen2.5-0.5B-Instruct GGUF (Q4量子化 / 約400MB) をダウンロード中...")
model_path = hf_hub_download(
    repo_id   = "Qwen/Qwen2.5-0.5B-Instruct-GGUF",
    filename  = "qwen2.5-0.5b-instruct-q4_k_m.gguf",
    local_dir = MODEL_DIR,
)
print(f"✅ {model_path}")

## Step 3 — LLMサーバーの起動 (llama-cpp-python)

In [ ]:
import subprocess, sys, time, requests, socket

LLM_PORT   = 8000
LLM_URL    = f"http://localhost:{LLM_PORT}"
gpu_layers = "32" if subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0 else "0"

llm_proc = subprocess.Popen(
    [
        sys.executable, "-m", "llama_cpp.server",
        "--model",        model_path,
        "--host",         "0.0.0.0",
        "--port",         str(LLM_PORT),
        "--n_gpu_layers", gpu_layers,
        "--n_ctx",        "2048",
        "--chat_format",  "chatml",
    ],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)

print("⏳ LLMサーバー起動待ち", end="")
for _ in range(60):
    try:
        if requests.get(f"{LLM_URL}/v1/models", timeout=2).status_code == 200:
            print(" ✅")
            print(f"🚀 LLMサーバー起動完了: {LLM_URL}  (GPU layers={gpu_layers})")
            break
    except Exception:
        pass
    print(".", end="", flush=True)
    time.sleep(2)
else:
    print("\n❌ タイムアウト")

## Step 4 — データソースの構築

### SQLite (RDB)
商品の構造化データ（ID・名前・カテゴリ・価格・在庫）を格納します。

### ChromaDB (ベクトルDB)
商品説明文を `sentence-transformers` で埋め込みベクトル化して格納します。
意味的に近い説明文をクエリーから検索できます。

In [ ]:
import sqlite3, os, json
import chromadb
import numpy as np
from sentence_transformers import SentenceTransformer

# ── サンプル商品データ ────────────────────────────────────────────────
PRODUCTS = [
    {"id": "P001", "name": "全自動コーヒーメーカー Pro",   "category": "キッチン家電", "price": 12800, "stock": 15,
     "description": "豆から全自動でコーヒーを淹れる本格派。タイマー予約付きで朝の一杯を自動準備。静音設計で早朝でも安心。"},
    {"id": "P002", "name": "コンパクトコーヒーメーカー",   "category": "キッチン家電", "price": 2980, "stock": 30,
     "description": "一人暮らしに最適なコンパクトサイズ。ドリップ式で手軽においしいコーヒーが楽しめる。"},
    {"id": "P003", "name": "電動ミルグラインダー",         "category": "キッチン家電", "price": 4500, "stock": 20,
     "description": "コーヒー豆を均一に挽けるコニカルバー式グラインダー。挽き目を15段階で調整可能。"},
    {"id": "P004", "name": "ノイズキャンセリングイヤホン", "category": "オーディオ",   "price": 8900, "stock": 25,
     "description": "最大30時間の連続再生とアクティブノイズキャンセリングで集中作業をサポート。"},
    {"id": "P005", "name": "ワイヤレスイヤホン 軽量",     "category": "オーディオ",   "price": 2500, "stock": 50,
     "description": "わずか4gの超軽量設計。スポーツや通勤に最適な防水IPX5対応のBluetoothイヤホン。"},
    {"id": "P006", "name": "Bluetoothスピーカー",         "category": "オーディオ",   "price": 5800, "stock": 18,
     "description": "360度サウンドの防水ポータブルスピーカー。屋外でもパワフルな低音を再現。"},
    {"id": "P007", "name": "メカニカルキーボード",         "category": "PC周辺機器",  "price": 9800, "stock": 12,
     "description": "青軸採用の打鍵感が心地よいフルサイズキーボード。RGBバックライト搭載。"},
    {"id": "P008", "name": "ワイヤレスマウス 静音",       "category": "PC周辺機器",  "price": 1980, "stock": 60,
     "description": "クリック音が静かな静音ワイヤレスマウス。USB-Cレシーバーで最大3台まで接続切替。"},
    {"id": "P009", "name": "ウルトラワイドモニター",       "category": "PC周辺機器",  "price": 45000, "stock": 5,
     "description": "34インチ湾曲ウルトラワイド。マルチタスク・動画編集・ゲームに最適な広大な作業領域。"},
    {"id": "P010", "name": "ハンディクリーナー",           "category": "生活家電",   "price": 3200, "stock": 22,
     "description": "コードレスで取り回し抜群。車内やデスク周りのゴミを素早く吸引。充電式で経済的。"},
]

# ── SQLite 構築 ───────────────────────────────────────────────────────
DB_PATH = "/content/products.db"
conn    = sqlite3.connect(DB_PATH)
cur     = conn.cursor()
cur.execute("DROP TABLE IF EXISTS products")
cur.execute("""
    CREATE TABLE products (
        id       TEXT PRIMARY KEY,
        name     TEXT,
        category TEXT,
        price    INTEGER,
        stock    INTEGER,
        description TEXT
    )
""")
for p in PRODUCTS:
    cur.execute(
        "INSERT INTO products VALUES (?,?,?,?,?,?)",
        (p["id"], p["name"], p["category"], p["price"], p["stock"], p["description"])
    )
conn.commit()
conn.close()
print(f"✅ SQLite 構築完了: {DB_PATH}  ({len(PRODUCTS)}件)")

# ── ChromaDB 構築 ─────────────────────────────────────────────────────
print("\n埋め込みモデルをロード中...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ 埋め込みモデル準備完了")

CHROMA_DIR = "/content/chroma"
chroma_client     = chromadb.PersistentClient(path=CHROMA_DIR)
# 既存コレクションがあれば削除して再作成
try:
    chroma_client.delete_collection("products")
except Exception:
    pass
collection = chroma_client.create_collection("products")

# 説明文をベクトル化して格納
descriptions = [p["description"] for p in PRODUCTS]
embeddings   = embedder.encode(descriptions, convert_to_numpy=True).tolist()
collection.add(
    ids        = [p["id"] for p in PRODUCTS],
    embeddings = embeddings,
    documents  = descriptions,
    metadatas  = [{"name": p["name"], "category": p["category"], "price": p["price"]} for p in PRODUCTS],
)
print(f"✅ ChromaDB 構築完了: {CHROMA_DIR}  ({len(PRODUCTS)}件)")

## Step 5 — 検索関数の定義

クエリーから以下の2種類の検索を行います。

| 検索 | 方法 | 向いているクエリー例 |
|---|---|---|
| RDB検索 | SQL WHERE句 | 「3000円以下」「在庫あり」「カテゴリ: オーディオ」|
| ベクトル検索 | コサイン類似度 | 「静かで軽いもの」「集中できる」「アウトドア向け」|

In [ ]:
import sqlite3, re
from typing import Optional

def search_rdb(
    category:  Optional[str] = None,
    max_price: Optional[int] = None,
    min_price: Optional[int] = None,
    in_stock:  bool = False,
    limit:     int  = 5,
) -> list[dict]:
    """
    SQLite から構造化条件で商品を検索する。
    条件なしの場合は全件を返す。
    """
    conn   = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    cur    = conn.cursor()
    wheres = []
    params = []
    if category:  wheres.append("category = ?");   params.append(category)
    if max_price: wheres.append("price <= ?");      params.append(max_price)
    if min_price: wheres.append("price >= ?");      params.append(min_price)
    if in_stock:  wheres.append("stock > 0")
    sql = "SELECT * FROM products"
    if wheres:
        sql += " WHERE " + " AND ".join(wheres)
    sql += f" ORDER BY price LIMIT {limit}"
    cur.execute(sql, params)
    rows = [dict(r) for r in cur.fetchall()]
    conn.close()
    return rows


def search_vector(query: str, top_k: int = 3) -> list[dict]:
    """
    ChromaDB でクエリーに意味的に近い商品説明文を検索する。
    """
    vec     = embedder.encode(query, convert_to_numpy=True).tolist()
    results = collection.query(
        query_embeddings = [vec],
        n_results        = top_k,
        include          = ["documents", "metadatas", "distances"],
    )
    items = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        items.append({
            "name":        meta["name"],
            "category":    meta["category"],
            "price":       meta["price"],
            "description": doc,
            "similarity":  round(1 - dist, 3),  # distance → similarity
        })
    return items


def merge_results(rdb_results: list, vec_results: list) -> list[dict]:
    """
    RDB結果とベクトル検索結果を名前をキーに重複排除してマージする。
    RDB結果を優先し、ベクトル結果で補完する。
    """
    seen  = set()
    merged = []
    for item in rdb_results:
        if item["name"] not in seen:
            item["source"] = "RDB"
            merged.append(item)
            seen.add(item["name"])
    for item in vec_results:
        if item["name"] not in seen:
            item["source"] = "ベクトルDB"
            merged.append(item)
            seen.add(item["name"])
    return merged


# 動作確認
print("=== RDB検索テスト: 価格3000円以下 ===")
for r in search_rdb(max_price=3000):
    print(f"  {r['name']}  ¥{r['price']}  [{r['category']}]")

print("\n=== ベクトル検索テスト: '静かで軽いイヤホン' ===")
for r in search_vector("静かで軽いイヤホン"):
    print(f"  {r['name']}  similarity={r['similarity']}")

## Step 6 — RAGサーバー (FastAPI) の起動

### エンドポイント
| メソッド | パス | 説明 |
|---|---|---|
| `POST` | `/ask` | 質問を受け取り、RAG+LLMで回答を返す |
| `GET` | `/products` | 商品一覧をフィルタ付きで返す |

In [ ]:
import asyncio, os, socket, threading, time
from contextlib import asynccontextmanager

import httpx
import nest_asyncio
import requests as req_lib
import uvicorn
from fastapi import FastAPI, Query
from pydantic import BaseModel

nest_asyncio.apply()
LLM_CHAT_URL = f"http://localhost:{LLM_PORT}/v1/chat/completions"

# ── スキーマ ─────────────────────────────────────────────────────────
class AskRequest(BaseModel):
    user_id: str
    question: str
    max_price: Optional[int] = None
    category:  Optional[str] = None

class AskResponse(BaseModel):
    user_id:      str
    answer:       str
    rdb_hits:     int
    vector_hits:  int
    sources:      list

# ── RAG + LLM 呼び出し ────────────────────────────────────────────────
async def rag_answer(req: AskRequest) -> AskResponse:
    # 1. RDB 検索
    rdb_results = search_rdb(
        category  = req.category,
        max_price = req.max_price,
        in_stock  = True,
    )
    # 2. ベクトル検索
    vec_results = search_vector(req.question, top_k=3)

    # 3. マージ
    merged = merge_results(rdb_results, vec_results)

    # 4. プロンプト組み立て
    context_lines = []
    for i, item in enumerate(merged, 1):
        context_lines.append(
            f"{i}. 【{item['name']}】"
            f" 価格: ¥{item['price']}"
            f" カテゴリ: {item['category']}"
            f" 取得元: {item.get('source','')}"
            f"\n   説明: {item.get('description', '')}"
        )
    context = "\n".join(context_lines) if context_lines else "該当商品なし"

    prompt = f"""以下の商品情報をもとに、ユーザーの質問に日本語で簡潔に答えてください。

【商品情報】
{context}

【ユーザーの質問】
{req.question}

回答:"""

    # 5. LLM へ直接 HTTP POST
    async with httpx.AsyncClient(timeout=120.0) as client:
        r = await client.post(
            LLM_CHAT_URL,
            json={
                "model": "local-qwen",
                "messages": [
                    {"role": "system", "content": "あなたは商品案内アシスタントです。提供された商品情報のみをもとに答えてください。"},
                    {"role": "user",   "content": prompt},
                ],
                "max_tokens": 300,
            },
        )
    answer = r.json()["choices"][0]["message"]["content"]

    return AskResponse(
        user_id     = req.user_id,
        answer      = answer,
        rdb_hits    = len(rdb_results),
        vector_hits = len(vec_results),
        sources     = [{"name": m["name"], "price": m["price"], "source": m.get("source","")} for m in merged],
    )

# ── FastAPI アプリ ────────────────────────────────────────────────────
@asynccontextmanager
async def lifespan(app: FastAPI):
    print("✅ RAGサーバー初期化完了")
    yield

app = FastAPI(title="RAGサーバー", version="1.0", lifespan=lifespan)

@app.post("/ask", response_model=AskResponse)
async def ask(req: AskRequest):
    return await rag_answer(req)

@app.get("/products")
async def get_products(
    category:  Optional[str] = Query(None),
    max_price: Optional[int] = Query(None),
    in_stock:  bool          = Query(False),
):
    return search_rdb(category=category, max_price=max_price, in_stock=in_stock, limit=20)

# ── 空きポートを取得して起動 ─────────────────────────────────────────
def get_free_port() -> int:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(('', 0))
        return s.getsockname()[1]

RAG_PORT = get_free_port()
RAG_URL  = f"http://localhost:{RAG_PORT}"
print(f"✅ RAGサーバー用ポート: {RAG_PORT}")

def run_rag_server():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    config = uvicorn.Config(app, host="0.0.0.0", port=RAG_PORT, log_level="warning", loop="none")
    server = uvicorn.Server(config)
    loop.run_until_complete(server.serve())

threading.Thread(target=run_rag_server, daemon=True).start()

print("⏳ RAGサーバー起動待ち", end="")
for _ in range(20):
    try:
        if req_lib.get(f"{RAG_URL}/products", timeout=2).status_code == 200:
            print(" ✅")
            print(f"🚀 RAGサーバー起動完了: {RAG_URL}")
            print(f"   API ドキュメント  : {RAG_URL}/docs")
            break
    except Exception:
        pass
    print(".", end="", flush=True)
    time.sleep(1)
else:
    print("\n❌ 起動タイムアウト")

## Step 7 — HTTPクライアントで質問を送る

実際のユーザーを模擬して `httpx` で `POST /ask` を送ります。

In [ ]:
import httpx, asyncio, json
from datetime import datetime

async def ask_rag(user_id: str, question: str,
                  max_price: int = None, category: str = None) -> dict:
    """RAGサーバーに質問を送り、回答を表示する"""
    ts = datetime.now().strftime("%H:%M:%S")
    print(f"\n[{ts}] {user_id}: {question}")
    if max_price: print(f"  フィルタ: 価格 <= ¥{max_price}")
    if category:  print(f"  フィルタ: カテゴリ = {category}")

    async with httpx.AsyncClient(timeout=120.0) as client:
        resp = await client.post(
            f"{RAG_URL}/ask",
            json={"user_id": user_id, "question": question,
                  "max_price": max_price, "category": category},
        )
    data = resp.json()

    print(f"\n  📚 取得コンテキスト ({len(data['sources'])}件: RDB={data['rdb_hits']} / ベクトルDB={data['vector_hits']})")
    for s in data["sources"]:
        print(f"    [{s['source']}] {s['name']}  ¥{s['price']}")
    print(f"\n  💬 回答:\n  {data['answer']}")
    print("  " + "─"*55)
    return data


async def run_demo():
    print("=" * 60)
    print("  RAG デモ: 3種類の質問パターン")
    print("=" * 60)

    # パターン1: 価格フィルタ + 意味検索
    await ask_rag(
        user_id   = "Alice",
        question  = "3000円以下でコーヒーを楽しめる商品はありますか？",
        max_price = 3000,
    )

    # パターン2: カテゴリフィルタ + 意味検索
    await ask_rag(
        user_id  = "Bob",
        question = "集中して作業できるオーディオ機器を探しています",
        category = "オーディオ",
    )

    # パターン3: フィルタなし（ベクトル検索のみ）
    await ask_rag(
        user_id  = "Carol",
        question = "アウトドアで使えるポータブルな製品はありますか？",
    )

asyncio.run(run_demo())

## Step 8 — 検索の内訳を確認する

RDB検索とベクトル検索それぞれが何を返しているか個別に確認する。

In [ ]:
question = "静かで使いやすいPC周辺機器を教えてください"

print(f"質問: {question}\n")

print("─" * 50)
print("【RDB検索】 カテゴリ: PC周辺機器")
for r in search_rdb(category="PC周辺機器"):
    print(f"  {r['name']:25s}  ¥{r['price']:>6,}  在庫:{r['stock']}")

print("\n" + "─" * 50)
print("【ベクトル検索】 意味的に近い商品 top3")
for r in search_vector(question, top_k=3):
    print(f"  similarity={r['similarity']:.3f}  {r['name']}")
    print(f"    → {r['description'][:60]}...")

print("\n" + "─" * 50)
print("【マージ後】")
merged = merge_results(search_rdb(category="PC周辺機器"), search_vector(question, top_k=3))
for m in merged:
    print(f"  [{m.get('source','')}] {m['name']}  ¥{m['price']}")

## アーキテクチャまとめ

```
ユーザー (httpx)
  ↕ HTTP POST /ask
    {user_id, question, max_price?, category?}
┌─────────────────────────────────────────────┐
│  RAGサーバー (FastAPI)                       │
│                                             │
│  1. RDB検索 (SQLite)                        │
│     WHERE category=? AND price<=? ...       │
│                                             │
│  2. ベクトル検索 (ChromaDB)                  │
│     question → 埋め込み → コサイン類似度     │
│                                             │
│  3. マージ (重複排除・RDB優先)               │
│                                             │
│  4. プロンプト組み立て                       │
│     「以下の商品情報をもとに答えて」          │
│      + マージ結果をコンテキストとして付与     │
└───────────────┬─────────────────────────────┘
                │ litellm.acompletion()
                │ HTTP POST /v1/chat/completions
┌───────────────▼─────────────────────────────┐
│  LLMサーバー (llama-cpp-python :8000)        │
│  Qwen2.5-0.5B-Instruct GGUF (Q4量子化)      │
└─────────────────────────────────────────────┘
```

### RDB と ベクトルDB の役割分担

| | RDB (SQLite) | ベクトルDB (ChromaDB) |
|---|---|---|
| 得意なクエリー | 「3000円以下」「在庫あり」 | 「静かなもの」「アウトドア向け」|
| 検索方法 | SQL WHERE句 (完全一致・範囲) | コサイン類似度 (意味的近さ) |
| 格納データ | 構造化データ (価格・カテゴリ等) | 説明文の埋め込みベクトル |
| 実運用での置き換え | PostgreSQL / MySQL | Pinecone / Weaviate / pgvector |